# Support Vector Machine (SVM)
## Real-world scenario: Classifying a tumour as benign or malignant

A clinic wants to classify a tumour as **malignant (1)** or **benign (0)** from two measurements (mean radius and mean texture). SVMs find the widest possible boundary between the two classes, which works well for medical classification tasks.

### Step 1 - Import the libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

np.random.seed(42)

### Step 2 - Create a small, realistic dataset
We simulate two groups of cells: benign tumours tend to be smaller, malignant ones larger.

In [ ]:
n_each = 30
# Benign: smaller radius & texture
benign_radius  = np.random.normal(12, 1.5, n_each)
benign_texture = np.random.normal(15, 2.0, n_each)
# Malignant: larger radius & texture
malig_radius   = np.random.normal(18, 2.0, n_each)
malig_texture  = np.random.normal(21, 2.5, n_each)

df = pd.DataFrame({
    'mean_radius':  np.concatenate([benign_radius, malig_radius]).round(2),
    'mean_texture': np.concatenate([benign_texture, malig_texture]).round(2),
    'malignant':    [0] * n_each + [1] * n_each
})

# Shuffle, then add messy data on purpose
df = df.sample(frac=1, random_state=1).reset_index(drop=True)
df.loc[5, 'mean_radius'] = np.nan
df = pd.concat([df, df.iloc[[0]]], ignore_index=True)
df.head()

### Step 3 - Explore the data

In [ ]:
print('Shape:', df.shape)
print('\nMissing:\n', df.isnull().sum())
print('\nDuplicates:', df.duplicated().sum())
print('\nClass balance:\n', df['malignant'].value_counts())

### Step 4 - Clean the data

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
df['mean_radius'] = df['mean_radius'].fillna(df['mean_radius'].median())
print('Missing after cleaning:', df.isnull().sum().sum())

### Step 5 - Features (X) and target (y)

In [ ]:
X = df[['mean_radius', 'mean_texture']]
y = df['malignant']

### Step 6 - Train / test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

### Step 7 - Scale the features
SVMs are distance-based, so scaling features to a common range is essential.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

### Step 8 - Train the SVM
We use an RBF kernel, which can separate classes that are not a straight line apart.

In [ ]:
model = SVC(kernel='rbf', C=1.0, random_state=42)
model.fit(X_train_scaled, y_train)

### Step 9 - Evaluate

In [ ]:
y_pred = model.predict(X_test_scaled)
print('Accuracy:', round(accuracy_score(y_test, y_pred), 3))
print('\nConfusion matrix:\n', confusion_matrix(y_test, y_pred))
print('\nReport:\n', classification_report(y_test, y_pred, zero_division=0))

### Step 10 - Predict for a new tumour
Mean radius 16, mean texture 19:

In [ ]:
new_tumour = pd.DataFrame({'mean_radius': [16], 'mean_texture': [19]})
new_scaled = scaler.transform(new_tumour)
pred = model.predict(new_scaled)[0]
print('Prediction:', 'Malignant' if pred == 1 else 'Benign')